In [ ]:
# =========================================================
# A2C COMPARATIVE ANALYSIS READY CODE
# FINAL UPDATED VERSION
# =========================================================

import numpy as np
import tkinter as tk
from tkinter import messagebox
import requests
import time
import psutil

GRID_SIZE = 16

ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]

gamma = 0.9
actor_lr = 0.01
critic_lr = 0.05

episodes = 3000

TILE_SIZE_CM = 50

start = None
goal = None
obstacles = []

mode = "obstacle"

# =========================================================
# COMPARATIVE ANALYSIS VARIABLES
# =========================================================

episode_rewards = []

steps_per_episode = []

success_count = 0

# =========================================================
# ACTOR POLICY
# =========================================================

policy = np.ones(
    (GRID_SIZE, GRID_SIZE, len(ACTIONS))
) / len(ACTIONS)

# =========================================================
# CRITIC VALUE TABLE
# =========================================================

V = np.zeros((GRID_SIZE, GRID_SIZE))

buttons = {}

tile_entry = None

# ---------------------------------------------------

def manhattan(a, b):

    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# ---------------------------------------------------

def softmax(x):

    exp_x = np.exp(x - np.max(x))

    return exp_x / np.sum(exp_x)

# ---------------------------------------------------

def step(state, action):

    r, c = state

    moves = {
        0: (-1, 0),
        1: (1, 0),
        2: (0, -1),
        3: (0, 1)
    }

    dr, dc = moves[action]

    nr = r + dr
    nc = c + dc

    # Wall collision

    if nr < 0 or nr >= GRID_SIZE or nc < 0 or nc >= GRID_SIZE:

        return state, -10, False

    new_state = (nr, nc)

    # Obstacle collision

    if new_state in obstacles:

        return new_state, -100, True

    # Goal

    if new_state == goal:

        return new_state, 500, True

    reward = -1

    # Distance shaping reward

    old_dist = manhattan(state, goal)

    new_dist = manhattan(new_state, goal)

    if new_dist < old_dist:
        reward += 5
    else:
        reward -= 5

    return new_state, reward, False

# ---------------------------------------------------

def train():

    global V, policy, TILE_SIZE_CM
    global success_count


    # RESET COMPARATIVE METRICS

    episode_rewards.clear()
    steps_per_episode.clear()
    success_count = 0

    # RESET A2C TABLES

    policy.fill(0.25)

    V.fill(0)
    if start is None or goal is None:

        messagebox.showerror(
            "Error",
            "Set Start and Goal first"
        )

        return

    TILE_SIZE_CM = float(tile_entry.get())

    # =====================================================
    # TRAINING TIMER START
    # =====================================================

    start_time = time.time()
    process = psutil.Process()

    start_cpu = process.cpu_times()
    
    start_ram = process.memory_info().rss / (1024 * 1024)
    # =====================================================

    for ep in range(episodes):

        state = start

        visited = set()

        total_reward = 0

        for step_count in range(300):

            r, c = state

            probs = softmax(policy[r, c])

            action = np.random.choice(
                len(ACTIONS),
                p=probs
            )

            new_state, reward, done = step(
                state,
                action
            )

            # Loop penalty

            if new_state in visited:
                reward -= 20
            else:
                visited.add(new_state)

            nr, nc = new_state

            # TD Target

            td_target = reward + gamma * V[nr, nc]

            # Advantage

            td_error = td_target - V[r, c]

            # Critic update

            V[r, c] += critic_lr * td_error

            # Actor update

            policy[r, c, action] += (
                actor_lr * td_error
            )

            state = new_state

            total_reward += reward

            if done:

                if state == goal:
                    success_count += 1

                break

        # =================================================
        # TRACK METRICS
        # =================================================

        episode_rewards.append(total_reward)

        steps_per_episode.append(step_count + 1)

    # =====================================================
    # TRAINING TIMER END
    # =====================================================

    end_time = time.time()

    training_time = end_time - start_time
    end_cpu = process.cpu_times()

    cpu_time = (
        (end_cpu.user + end_cpu.system)
        -
        (start_cpu.user + start_cpu.system)
    )
    
    end_ram = process.memory_info().rss / (1024 * 1024)
    
    ram_usage = end_ram - start_ram
    
    policy_memory = (
        policy.nbytes / (1024 * 1024)
    )
    
    value_memory = (
        V.nbytes / (1024 * 1024)
    )
    
    model_memory = (
        policy_memory + value_memory
    )
    # =====================================================
    # RESOURCE USAGE
    # =====================================================

   
    # =====================================================
    # PERFORMANCE METRICS
    # =====================================================

    avg_reward = np.mean(episode_rewards)

    avg_steps = np.mean(steps_per_episode)

    success_rate = (
        success_count / episodes
    ) * 100

    # =====================================================
    # FINAL RESULTS
    # =====================================================

    print("\n==============================")
    print("A2C COMPARATIVE RESULTS")
    print("==============================")

    print(f"Training Time: {training_time:.2f} sec")

    print(f"CPU Time: {cpu_time:.2f} sec")

    print(
        f"Training Memory Increase: "
        f"{ram_usage:.2f} MB"
    )
    
    print(
        f"Model Memory: "
        f"{model_memory:.4f} MB"
    )

    print(f"Average Reward: {avg_reward:.2f}")

    print(f"Average Steps: {avg_steps:.2f}")

    print(f"Success Rate: {success_rate:.2f}%")

    print("==============================")

    print("\nVALUE TABLE\n")

    print(V)

    final_policy = extract_policy()

    simulate_path(final_policy)

    send_policy(final_policy)

# ---------------------------------------------------

def extract_policy():

    final_policy = {}

    print("\nPOLICY\n")

    for r in range(GRID_SIZE):

        for c in range(GRID_SIZE):

            state = (r, c)

            if state in obstacles:
                continue

            if state == goal:

                final_policy[str(state)] = "GOAL"

                print(state, "-> GOAL")

                continue

            action = np.argmax(policy[r, c])

            final_policy[str(state)] = ACTIONS[action]

            print(
                state,
                "->",
                ACTIONS[action]
            )

    return final_policy

# ---------------------------------------------------

def simulate_path(policy_map):

    global TILE_SIZE_CM

    state = start

    path = [state]

    steps = 0

    visited = set()

    for _ in range(500):

        if state == goal:
            break

        if state in visited:

            print("Loop detected")

            break

        visited.add(state)

        action = policy_map[str(state)]

        r, c = state

        if action == "UP":
            r -= 1

        elif action == "DOWN":
            r += 1

        elif action == "LEFT":
            c -= 1

        elif action == "RIGHT":
            c += 1

        state = (r, c)

        path.append(state)

        steps += 1

    distance = steps * TILE_SIZE_CM

    print("\nPATH:")

    print(path)

    print("\nSteps:", steps)

    print("Tile Size:", TILE_SIZE_CM, "cm")

    print("Total Distance:", distance, "cm")

# ---------------------------------------------------

def send_policy(policy_map):

    state = start

    actions = []

    visited = set()

    for _ in range(500):

        if state == goal:

            actions.append("GOAL")

            break

        if state in visited:
            break

        visited.add(state)

        action = policy_map[str(state)]

        actions.append(action)

        r, c = state

        if action == "UP":
            r -= 1

        elif action == "DOWN":
            r += 1

        elif action == "LEFT":
            c -= 1

        elif action == "RIGHT":
            c += 1

        state = (r, c)

    data = {
        "tile_size": TILE_SIZE_CM,
        "path": actions
    }

    url = "http://192.168.4.1/policy"

    try:

        requests.post(url, json=data)

        print("\nPath sent to ESP32")

        print(actions)

    except:

        print("\nESP32 not connected")

# ---------------------------------------------------

def cell_click(r, c):

    global start, goal

    if mode == "start":

        if start:
            buttons[start].config(
                bg="white",
                text=""
            )

        start = (r, c)

        buttons[(r, c)].config(
            bg="blue",
            text="S",
            fg="white"
        )

    elif mode == "goal":

        if goal:
            buttons[goal].config(
                bg="white",
                text=""
            )

        goal = (r, c)

        buttons[(r, c)].config(
            bg="green",
            text="G",
            fg="white"
        )

    elif mode == "obstacle":

        if (r, c) not in obstacles:
            obstacles.append((r, c))

        buttons[(r, c)].config(
            bg="black",
            text="X",
            fg="white"
        )

    elif mode == "erase":

        if (r, c) in obstacles:
            obstacles.remove((r, c))

        if start == (r, c):
            start = None

        if goal == (r, c):
            goal = None

        buttons[(r, c)].config(
            bg="white",
            text=""
        )

# ---------------------------------------------------

def set_mode(m):

    global mode

    mode = m

# ---------------------------------------------------

def build_gui():

    global tile_entry

    root = tk.Tk()

    root.title("16x16 A2C Robot Trainer")

    control = tk.Frame(root)

    control.pack()

    tk.Button(
        control,
        text="Set Start",
        command=lambda: set_mode("start")
    ).grid(row=0, column=0)

    tk.Button(
        control,
        text="Set Goal",
        command=lambda: set_mode("goal")
    ).grid(row=0, column=1)

    tk.Button(
        control,
        text="Add Obstacle",
        command=lambda: set_mode("obstacle")
    ).grid(row=0, column=2)

    tk.Button(
        control,
        text="Erase",
        command=lambda: set_mode("erase")
    ).grid(row=0, column=3)

    tk.Button(
        control,
        text="Train A2C",
        command=train
    ).grid(row=0, column=4)

    tk.Label(
        control,
        text="Tile Size (cm)"
    ).grid(row=1, column=0)

    tile_entry = tk.Entry(control, width=10)

    tile_entry.insert(0, "40")

    tile_entry.grid(row=1, column=1)

    grid_frame = tk.Frame(root)

    grid_frame.pack()

    for r in range(GRID_SIZE):

        for c in range(GRID_SIZE):

            b = tk.Button(
                grid_frame,
                width=3,
                height=1,
                bg="white",
                command=lambda r=r, c=c:
                cell_click(r, c)
            )

            b.grid(row=r, column=c)

            buttons[(r, c)] = b

    root.mainloop()

# ---------------------------------------------------

build_gui()